### author by yangshichen
### 注意：脚本仅供参考，使用前请仔细阅读

In [1]:
library(ggplot2)
library(future)
library(tidyverse)
library(ggpubr)
library(ggchicklet)
library(ggsci)
library(magrittr)
library(ggh4x)
library(rstatix)
library(ggsignif)
library(ggnewscale)
library(patchwork)
library(gapminder)
library(ggprism)
library(dplyr)
library(ggplotify)
library(readr)
library(arrow)
library(ggbreak)
library(data.table)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.2
✔ lubridate 1.9.3     ✔ tibble    3.2.1
✔ purrr     1.0.2     ✔ tidyr     1.3.1
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

载入程序包：‘magrittr’


The following object is masked from ‘package:purrr’:

    set_names


The following object is masked from ‘package:tidyr’:

    extract



载入程序包：‘rstatix’


The following object is masked from ‘package:stats’:

    filter



载入程序包：‘arrow’


The following object is masked from ‘package:magrittr’:

    is_in


The following object is masked from ‘package:lubridate’:

    duration


The following object is masked from ‘package:utils’:

    timestamp


ggbreak v0.1.5 Learn more 

In [2]:
mytheme <- theme_prism(base_family="",base_fontface="plain") +
        theme(strip.text = element_text(size = 8,angle=10,vjust = 0.5,hjust = 0.5),
        axis.line = element_line(color = "black",size = 0.2),
        axis.ticks = element_line(size = 0.2),
        axis.text.y = element_text(color = "black",size = 6),
        axis.text.x = element_text(color = "black",size = 6, angle = 30,hjust = 1,vjust = 1),
        axis.title = element_text(color = "black",size = 10),
        legend.position = "none")

Warning message:
“The `size` argument of `element_line()` is deprecated as of ggplot2 3.4.0.
ℹ Please use the `linewidth` argument instead.”


### eQTL-pQTL-1

In [37]:
SMR_dir <- "/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Result/xQTL_SMR/"

# 获取所有 .smr 文件路径（包括子目录）
smr_files <- list.files(SMR_dir, pattern = "\\.smr$", recursive = TRUE, full.names = TRUE)

# 检查文件数
cat("Found", length(smr_files), "SMR result files.\n")

Found 99 SMR result files.


In [40]:
# 读取并合并
smr_list <- lapply(smr_files, function(f) {
  df <- tryCatch({
    read.table(f, header = TRUE, sep = "\t",
               stringsAsFactors = FALSE,
               colClasses = "character")
  }, error = function(e) {
    message("Error reading ", f)
    return(NULL)
  })
  
  if (!is.null(df) && nrow(df) > 0) {
    df$celltype <- tools::file_path_sans_ext(basename(f))
  }
  
  return(df)
})

In [41]:
smr_all <- bind_rows(smr_list)
smr_all$p_Outco <- as.numeric(smr_all$p_Outco)
smr_all$p_Expo <- as.numeric(smr_all$p_Expo)
smr_all$p_SMR <- as.numeric(smr_all$p_SMR)
smr_all$q_SMR <- p.adjust(smr_all$p_SMR, method = "BH")
smr_all$p_HEIDI <- as.numeric(smr_all$p_HEIDI)
smr_all

Expo_ID,Expo_Chr,Expo_Gene,Expo_bp,Outco_ID,Outco_Chr,Outco_Gene,Outco_bp,topSNP,topSNP_chr,⋯,b_Expo,se_Expo,p_Expo,b_SMR,se_SMR,p_SMR,p_HEIDI,nsnp_HEIDI,celltype,q_SMR
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<dbl>
TNFRSF9,1,TNFRSF9,7915871,CAMTA1,1,CAMTA1,6785454,1:7931143:G:A,1,⋯,0.080948,0.0112921,7.579456e-13,0.63276,0.537871,2.394293e-01,8.789142e-01,20,Adaptive NK cells,6.163414e-01
TNFRSF9,1,TNFRSF9,7915871,VAMP3,1,VAMP3,7771296,1:7931143:G:A,1,⋯,0.080948,0.0112921,7.579456e-13,1.50138,0.556408,6.968435e-03,3.461237e-01,20,Adaptive NK cells,5.219059e-02
TNFRSF9,1,TNFRSF9,7915871,UTS2,1,UTS2,7843083,1:7931143:G:A,1,⋯,0.080948,0.0112921,7.579456e-13,-10.4726,1.56857,2.447473e-11,5.282286e-02,20,Adaptive NK cells,8.652388e-10
TNFRSF9,1,TNFRSF9,7915871,TNFRSF9,1,TNFRSF9,7915871,1:7931143:G:A,1,⋯,0.080948,0.0112921,7.579456e-13,7.05589,1.12663,3.780010e-10,1.439752e-01,20,Adaptive NK cells,9.277341e-09
TNFRSF9,1,TNFRSF9,7915871,PARK7,1,PARK7,7954291,1:7931143:G:A,1,⋯,0.080948,0.0112921,7.579456e-13,0.204577,0.455708,6.534891e-01,2.074170e-08,20,Adaptive NK cells,8.963680e-01
TNFRSF9,1,TNFRSF9,7915871,RERE,1,RERE,8352397,1:7931143:G:A,1,⋯,0.080948,0.0112921,7.579456e-13,-0.0311607,0.412891,9.398411e-01,4.443957e-02,20,Adaptive NK cells,9.848862e-01
TNFRSF9,1,TNFRSF9,7915871,ENO1,1,ENO1,8861000,1:7931143:G:A,1,⋯,0.080948,0.0112921,7.579456e-13,0.531376,0.439084,2.262052e-01,8.935655e-02,20,Adaptive NK cells,5.996108e-01
CD6,11,CD6,60971680,MS4A7,11,MS4A7,60378485,11:61008737:C:T,11,⋯,-0.214477,0.0191004,2.941038e-29,0.224492,0.231119,3.313844e-01,3.541619e-01,17,Adaptive NK cells,7.024659e-01
CD5,11,CD5,61102489,MS4A7,11,MS4A7,60378485,11:61008737:C:T,11,⋯,0.151929,0.0152231,1.860913e-23,-0.316913,0.326592,3.318650e-01,2.539811e-01,13,Adaptive NK cells,7.032034e-01


In [42]:
colnames(smr_all)

[1] "Expo_ID"    "Expo_Chr"   "Expo_Gene"  "Expo_bp"    "Outco_ID"  
 [6] "Outco_Chr"  "Outco_Gene" "Outco_bp"   "topSNP"     "topSNP_chr"
[11] "topSNP_bp"  "A1"         "A2"         "Freq"       "b_Outco"   
[16] "se_Outco"   "p_Outco"    "b_Expo"     "se_Expo"    "p_Expo"    
[21] "b_SMR"      "se_SMR"     "p_SMR"      "p_HEIDI"    "nsnp_HEIDI"
[26] "celltype"   "q_SMR"

In [43]:
sub_smr_all <- subset(smr_all, p_Outco < 0.00001 & p_HEIDI > 0.01 & q_SMR < 0.05 & p_Expo < 0.00000005)
sub_smr_all

,Expo_ID,Expo_Chr,Expo_Gene,Expo_bp,Outco_ID,Outco_Chr,Outco_Gene,Outco_bp,topSNP,topSNP_chr,⋯,b_Expo,se_Expo,p_Expo,b_SMR,se_SMR,p_SMR,p_HEIDI,nsnp_HEIDI,celltype,q_SMR
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<dbl>
3,TNFRSF9,1,TNFRSF9,7915871,UTS2,1,UTS2,7843083,1:7931143:G:A,1,⋯,0.080948,0.0112921,7.579456e-13,-10.4726,1.56857,2.447473e-11,0.05282286,20,Adaptive NK cells,8.652388e-10
4,TNFRSF9,1,TNFRSF9,7915871,TNFRSF9,1,TNFRSF9,7915871,1:7931143:G:A,1,⋯,0.080948,0.0112921,7.579456e-13,7.05589,1.12663,3.780010e-10,0.14397520,20,Adaptive NK cells,9.277341e-09
24,SULT1A1,16,SULT1A1,28605196,SULT1A1,16,SULT1A1,28605196,16:28608999:G:A,16,⋯,0.126989,0.0165622,1.755043e-14,4.05286,0.654468,5.917967e-10,0.98886960,20,Adaptive NK cells,1.402024e-08
82,CCL3,17,CCL3,36088256,CCL3-AS1,17,CCL3-AS1,36072866,17:36105010:C:A,17,⋯,0.172997,0.0111452,2.455842e-54,-1.49036,0.264405,1.733683e-08,0.02534640,20,Adaptive NK cells,3.370062e-07
83,CCL4,17,CCL4,36103827,CCL3-AS1,17,CCL3-AS1,36072866,17:36105010:C:A,17,⋯,-0.183634,0.0127089,2.537605e-47,1.40404,0.251607,2.401299e-08,0.03700546,20,Adaptive NK cells,4.551129e-07
88,CCL3,17,CCL3,36088256,CCL4,17,CCL4,36103827,17:36105010:C:A,17,⋯,0.172997,0.0111452,2.455842e-54,1.34701,0.230743,5.292269e-09,0.20745880,20,Adaptive NK cells,1.094217e-07
89,CCL4,17,CCL4,36103827,CCL4,17,CCL4,36103827,17:36105010:C:A,17,⋯,-0.183634,0.0127089,2.537605e-47,-1.26899,0.219734,7.689017e-09,0.21336340,20,Adaptive NK cells,1.538480e-07
137,IL18R1,2,IL18R1,102311529,IL18R1,2,IL18R1,102311529,2:102368964:A:G,2,⋯,0.369964,0.023101,1.002592e-57,1.23784,0.123092,8.621470e-24,0.03016855,20,Adaptive NK cells,2.100867e-21
226,TNFRSF9,1,TNFRSF9,7915871,TNFRSF9,1,TNFRSF9,7915871,1:7931143:G:A,1,⋯,0.080948,0.0112921,7.579456e-13,3.78913,0.708211,8.781894e-08,0.16559030,20,ALPL- MARCKS- NDNs,1.556334e-06


In [44]:
tmp <- subset(sub_smr_all, Expo_ID == Outco_ID)
tmp

,Expo_ID,Expo_Chr,Expo_Gene,Expo_bp,Outco_ID,Outco_Chr,Outco_Gene,Outco_bp,topSNP,topSNP_chr,⋯,b_Expo,se_Expo,p_Expo,b_SMR,se_SMR,p_SMR,p_HEIDI,nsnp_HEIDI,celltype,q_SMR
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<dbl>
4,TNFRSF9,1,TNFRSF9,7915871,TNFRSF9,1,TNFRSF9,7915871,1:7931143:G:A,1,⋯,0.080948,0.0112921,7.579456e-13,7.05589,1.12663,3.780010e-10,0.14397520,20,Adaptive NK cells,9.277341e-09
24,SULT1A1,16,SULT1A1,28605196,SULT1A1,16,SULT1A1,28605196,16:28608999:G:A,16,⋯,0.126989,0.0165622,1.755043e-14,4.05286,0.654468,5.917967e-10,0.98886960,20,Adaptive NK cells,1.402024e-08
89,CCL4,17,CCL4,36103827,CCL4,17,CCL4,36103827,17:36105010:C:A,17,⋯,-0.183634,0.0127089,2.537605e-47,-1.26899,0.219734,7.689017e-09,0.21336340,20,Adaptive NK cells,1.538480e-07
137,IL18R1,2,IL18R1,102311529,IL18R1,2,IL18R1,102311529,2:102368964:A:G,2,⋯,0.369964,0.023101,1.002592e-57,1.23784,0.123092,8.621470e-24,0.03016855,20,Adaptive NK cells,2.100867e-21
226,TNFRSF9,1,TNFRSF9,7915871,TNFRSF9,1,TNFRSF9,7915871,1:7931143:G:A,1,⋯,0.080948,0.0112921,7.579456e-13,3.78913,0.708211,8.781894e-08,0.16559030,20,ALPL- MARCKS- NDNs,1.556334e-06
257,SULT1A1,16,SULT1A1,28605196,SULT1A1,16,SULT1A1,28605196,16:28608999:G:A,16,⋯,0.126989,0.0165622,1.755043e-14,5.7094,0.835997,8.523980e-12,0.86939730,20,ALPL- MARCKS- NDNs,3.304495e-10
302,CCL4,17,CCL4,36103827,CCL4,17,CCL4,36103827,17:36105010:C:A,17,⋯,-0.183634,0.0127089,2.537605e-47,-0.966826,0.209859,4.084709e-06,0.29968130,20,ALPL- MARCKS- NDNs,5.770180e-05
482,IL18R1,2,IL18R1,102311529,IL18R1,2,IL18R1,102311529,2:102368964:A:G,2,⋯,0.369964,0.023101,1.002592e-57,-0.503789,0.117906,1.930339e-05,0.66223460,20,Basophils,2.503936e-04
520,IL18R1,2,IL18R1,102311529,IL18R1,2,IL18R1,102311529,2:102368964:A:G,2,⋯,0.369964,0.023101,1.002592e-57,-0.555382,0.12556,9.722856e-06,0.01700449,20,CCR4+ CD8+ Tcm,1.318868e-04


In [45]:
write.csv(sub_smr_all,'/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Result/SMR_eQTL_pQTL_1.csv')
write.csv(tmp,'/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Result/SMR_eQTL_pQTL_1_same.csv')

### eQTL-GWAS

In [3]:
SMR_dir <- "/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Result/eQTL_GWAS_SMR/"
disease_dirs <- list.dirs(SMR_dir, full.names = TRUE, recursive = FALSE)
all_smr_list <- list()

In [4]:
disease_dirs

[1] "/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Result/eQTL_GWAS_SMR//110"                                    
  [2] "/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Result/eQTL_GWAS_SMR//110.1"                                  
  [3] "/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Result/eQTL_GWAS_SMR//110.11"                                 
  [4] "/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Result/eQTL_GWAS_SMR//110.12"                                 
  [5] "/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Result/eQTL_GWAS_SMR//110.13"                                 
  [6] "/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Result/eQTL_GWAS_SMR//274"                                    
  [7] "/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Result/eQTL_GWAS_SMR//274.1"                                  
  [8] "/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Result/eQTL_GWAS_SMR//274.11"                                 
  [9] "/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Result/eQTL_GWAS_SMR//371"                                    
 [10] "/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Result/eQTL_GWAS_SMR//371.3"                                  
 [11] "/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Result/eQTL_GWAS_SMR//380.1"                                  
 [12] "/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Result/eQTL_GWAS_SMR//381"                                    
 [13] "/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Result/eQTL_GWAS_SMR//381.1"                                  
 [14] "/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Result/eQTL_GWAS_SMR//381.11"                                 
 [15] "/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Result/eQTL_GWAS_SMR//411.1"                                  
 [16] "/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Result/eQTL_GWAS_SMR//411.2"                                  
 [17] "/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Result/eQTL_GWAS_SMR//411.3"                                  
 [18] "/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Result/eQTL_GWAS_SMR//426"                                    
 [19] "/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Result/eQTL_GWAS_SMR//427"                                    
 [20] "/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Result/eQTL_GWAS_SMR//427.1"                                  
 [21] "/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Result/eQTL_GWAS_SMR//427.12"                                 
 [22] "/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Result/eQTL_GWAS_SMR//427.2"                                  
 [23] "/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Result/eQTL_GWAS_SMR//427.21"                                 
 [24] "/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Result/eQTL_GWAS_SMR//427.3"                                  
 [25] "/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Result/eQTL_GWAS_SMR//427.5"                                  
 [26] "/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Result/eQTL_GWAS_SMR//465"                                    
 [27] "/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Result/eQTL_GWAS_SMR//483"                                    
 [28] "/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Result/eQTL_GWAS_SMR//495"                                    
 [29] "/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Result/eQTL_GWAS_SMR//496.21"                                 
 [30] "/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Result/eQTL_GWAS_SMR//512.8"                                  
 [31] "/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Result/eQTL_GWAS_SMR//522.1"                                  
 [32] "/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Result/eQTL_GWAS_SMR//523"                                    
 [33] "/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Result/eQTL_GWAS_SMR//530.1"                                  
 [34] "/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Result/eQTL_GWAS_SMR//530.11"                                 
 [35] "/media/AnalysisDi

In [5]:
# 遍历每个疾病文件夹
for (disease_path in disease_dirs) {
  disease_name <- basename(disease_path)  # 疾病名
  # 找出该疾病下所有smr文件
  smr_files <- list.files(disease_path, pattern = "\\.smr$", full.names = TRUE)
  if (length(smr_files) == 0) next
  # 读取每个smr文件
  for (f in smr_files) {
    # 提取细胞类型名（假设文件名类似 CD4_Tcells.smr 或 Mono.smr）
    celltype <- str_remove(basename(f), "\\.smr$")
    # 读取smr文件
    smr_df <- tryCatch({
      read.table(f, header = TRUE, sep = "\t", stringsAsFactors = FALSE, colClasses = "character")
    }, error = function(e) {
      message("Failed to read ", f)
      return(NULL)
    })
    # 跳过空文件
    if (is.null(smr_df) || nrow(smr_df) == 0) next
    # 添加疾病和细胞类型信息
    smr_df$disease <- disease_name
    smr_df$celltype <- celltype
    # 存入列表
    all_smr_list[[length(all_smr_list) + 1]] <- smr_df
  }
}

In [6]:
all_smr <- rbindlist(all_smr_list, fill = TRUE)
all_smr

probeID,ProbeChr,Gene,Probe_bp,topSNP,topSNP_chr,topSNP_bp,A1,A2,Freq,⋯,b_eQTL,se_eQTL,p_eQTL,b_SMR,se_SMR,p_SMR,p_HEIDI,nsnp_HEIDI,disease,celltype
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
LINC00115,1,LINC00115,824228,1:913469:G:C,1,913469,C,G,0.102594,⋯,0.754036,0.078988,1.345376e-21,-0.0425695,0.0303973,1.613834e-01,9.389883e-01,20,110,eQTLsmr_Adaptive NK cells
LINC01128,1,LINC01128,825138,1:913469:G:C,1,913469,C,G,0.102594,⋯,0.965222,0.0741122,8.960174e-39,-0.0332554,0.023628,1.592911e-01,9.789697e-01,20,110,eQTLsmr_Adaptive NK cells
HES4,1,HES4,998962,1:1000453:C:G,1,1000453,C,G,0.126179,⋯,0.580657,0.0654351,7.069643e-19,0.0451232,0.0331989,1.740902e-01,6.294022e-01,20,110,eQTLsmr_Adaptive NK cells
ISG15,1,ISG15,1001138,1:1015903:C:T,1,1015903,C,T,0.231132,⋯,0.190143,0.0308707,7.304699e-10,0.096067,0.0774727,2.149712e-01,7.739581e-01,20,110,eQTLsmr_Adaptive NK cells
TNFRSF18,1,TNFRSF18,1203508,1:1217733:G:A,1,1217733,A,G,0.268278,⋯,-0.721839,0.0438161,5.615836e-61,0.0248688,0.0203157,2.209064e-01,2.585718e-01,20,110,eQTLsmr_Adaptive NK cells
TNFRSF4,1,TNFRSF4,1211326,1:1217251:C:A,1,1217251,A,C,0.266509,⋯,-0.653871,0.0426808,5.620573e-53,0.0299354,0.0224409,1.822141e-01,2.946277e-01,20,110,eQTLsmr_Adaptive NK cells
SDF4,1,SDF4,1216931,1:1231507:T:C,1,1231507,C,T,0.23467,⋯,-0.327702,0.0398609,2.015586e-16,0.0829247,0.0513187,1.061209e-01,4.074960e-01,20,110,eQTLsmr_Adaptive NK cells
B3GALT6,1,B3GALT6,1232237,1:1235869:T:C,1,1235869,C,T,0.224646,⋯,-0.359584,0.0399259,2.131788e-19,0.0918693,0.0473614,5.241006e-02,6.053866e-01,20,110,eQTLsmr_Adaptive NK cells
CPTP,1,CPTP,1324756,1:1363031:C:T,1,1363031,C,T,0.17158,⋯,0.810526,0.0464073,2.624344e-68,0.0122949,0.0201205,5.411574e-01,3.976492e-01,20,110,eQTLsmr_Adaptive NK cells


In [7]:
colnames(all_smr)

[1] "probeID"    "ProbeChr"   "Gene"       "Probe_bp"   "topSNP"    
 [6] "topSNP_chr" "topSNP_bp"  "A1"         "A2"         "Freq"      
[11] "b_GWAS"     "se_GWAS"    "p_GWAS"     "b_eQTL"     "se_eQTL"   
[16] "p_eQTL"     "b_SMR"      "se_SMR"     "p_SMR"      "p_HEIDI"   
[21] "nsnp_HEIDI" "disease"    "celltype"

In [8]:
all_smr$p_GWAS <- as.numeric(all_smr$p_GWAS)
all_smr$p_eQTL <- as.numeric(all_smr$p_eQTL)
all_smr$p_SMR <- as.numeric(all_smr$p_SMR)
all_smr <- all_smr %>% group_by(disease) %>% mutate(q_SMR = p.adjust(p_SMR, method = "BH")) %>% ungroup()
all_smr$p_HEIDI <- as.numeric(all_smr$p_HEIDI)

In [9]:
min(all_smr$q_SMR)

[1] 6.463894e-63

In [10]:
sub_smr_all <- subset(all_smr, p_GWAS < 0.00001 & p_HEIDI > 0.01 & q_SMR < 0.05 & p_eQTL < 0.00000005)
sub_smr_all

probeID,ProbeChr,Gene,Probe_bp,topSNP,topSNP_chr,topSNP_bp,A1,A2,Freq,⋯,se_eQTL,p_eQTL,b_SMR,se_SMR,p_SMR,p_HEIDI,nsnp_HEIDI,disease,celltype,q_SMR
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<dbl>
WDR1,4,WDR1,10068089,4:10113515:G:A,4,10113515,A,G,0.373231,⋯,0.0269378,2.049959e-11,0.439318,0.0896947,9.685428e-07,0.01504522,20,274,eQTLsmr_CD56dim NK cells,3.474427e-03
ABCG2,4,ABCG2,88090150,4:88143450:C:T,4,88143450,C,T,0.214623,⋯,0.0556689,6.954282e-09,0.675561,0.123167,4.136912e-08,0.35817250,20,274,eQTLsmr_Core NDNs,1.920501e-04
WDR1,4,WDR1,10068089,4:10105174:C:T,4,10105174,T,C,0.317217,⋯,0.0335252,3.572675e-11,0.418898,0.0813193,2.587186e-07,0.05148793,20,274,eQTLsmr_GZMB+ CD8+ Tem,1.074635e-03
WDR1,4,WDR1,10068089,4:10072546:A:T,4,10072546,T,A,0.474057,⋯,0.02637,2.400999e-10,0.565424,0.109266,2.282152e-07,0.07245705,20,274,eQTLsmr_HLA-DRhi CD8+ Tem,1.000597e-03
ABCG2,4,ABCG2,88090150,4:88205495:C:T,4,88205495,C,T,0.416863,⋯,0.0485434,3.064873e-09,0.588746,0.105879,2.688908e-08,0.06904124,20,274,eQTLsmr_Tfh,1.326304e-04
WDR1,4,WDR1,10068089,4:10113515:G:A,4,10113515,A,G,0.373231,⋯,0.0269378,2.049959e-11,0.441534,0.0900136,9.333410e-07,0.01504935,20,274.1,eQTLsmr_CD56dim NK cells,3.348149e-03
ABCG2,4,ABCG2,88090150,4:88143450:C:T,4,88143450,C,T,0.214623,⋯,0.0556689,6.954282e-09,0.671089,0.122463,4.254331e-08,0.34679310,20,274.1,eQTLsmr_Core NDNs,1.975011e-04
WDR1,4,WDR1,10068089,4:10105174:C:T,4,10105174,T,C,0.317217,⋯,0.0335252,3.572675e-11,0.419848,0.0814952,2.579774e-07,0.05228874,20,274.1,eQTLsmr_GZMB+ CD8+ Tem,1.071557e-03
WDR1,4,WDR1,10068089,4:10072546:A:T,4,10072546,T,A,0.474057,⋯,0.02637,2.400999e-10,0.567361,0.109584,2.250008e-07,0.07100712,20,274.1,eQTLsmr_HLA-DRhi CD8+ Tem,9.865035e-04


In [11]:
table(sub_smr_all$disease)


                                  274                                 274.1 
                                    5                                     5 
                               274.11                                   580 
                                    8                                     1 
                                  696                                 696.4 
                                  201                                   204 
                               696.41                                 714.1 
                                  192                                    56 
                                   AD                                    As 
                                  174                                   298 
                               Asthma                     Atopic_dermatitis 
                                  250                                    76 
                        Breast_cancer                                   CHB

In [12]:
write.csv(sub_smr_all,'/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Result/SMR_eQTL_disease.csv')

### eQTL-pQTL2

In [18]:
SMR_dir <- "/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Result/eQTL_pQTL_SMR/"
disease_dirs <- list.dirs(SMR_dir, full.names = TRUE, recursive = FALSE)
all_smr_list <- list()

In [19]:
# 遍历每个疾病文件夹
for (disease_path in disease_dirs) {
  disease_name <- basename(disease_path)  # 疾病名
  # 找出该疾病下所有smr文件
  smr_files <- list.files(disease_path, pattern = "\\.smr$", full.names = TRUE)
  if (length(smr_files) == 0) next
  # 读取每个smr文件
  for (f in smr_files) {
    # 提取细胞类型名（假设文件名类似 CD4_Tcells.smr 或 Mono.smr）
    celltype <- str_remove(basename(f), "\\.smr$")
    # 读取smr文件
    smr_df <- tryCatch({
      read.table(f, header = TRUE, sep = "\t", stringsAsFactors = FALSE, colClasses = "character")
    }, error = function(e) {
      message("Failed to read ", f)
      return(NULL)
    })
    # 跳过空文件
    if (is.null(smr_df) || nrow(smr_df) == 0) next
    # 添加疾病和细胞类型信息
    smr_df$disease <- disease_name
    smr_df$celltype <- celltype
    # 存入列表
    all_smr_list[[length(all_smr_list) + 1]] <- smr_df
  }
}

In [20]:
all_smr <- rbindlist(all_smr_list, fill = TRUE)
all_smr

probeID,ProbeChr,Gene,Probe_bp,topSNP,topSNP_chr,topSNP_bp,A1,A2,Freq,⋯,b_eQTL,se_eQTL,p_eQTL,b_SMR,se_SMR,p_SMR,p_HEIDI,nsnp_HEIDI,disease,celltype
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
LINC00115,1,LINC00115,824228,1:928209:C:T,1,928209,T,C,0.106132,⋯,0.675476,0.0736853,4.862338e-20,-0.330138,0.209492,1.150494e-01,2.657829e-02,16,adenosine_deaminase_measurement,pQTLsmr_Adaptive NK cells
LINC01128,1,LINC01128,825138,1:928209:C:T,1,928209,T,C,0.106132,⋯,0.84287,0.069478,7.195198e-34,-0.264572,0.166819,1.127430e-01,2.984633e-02,18,adenosine_deaminase_measurement,pQTLsmr_Adaptive NK cells
HES4,1,HES4,998962,1:1000112:G:T,1,1000112,G,T,0.126179,⋯,0.580657,0.0654351,7.069643e-19,-0.0247995,0.0270111,3.585545e-01,1.589803e-01,11,adenosine_deaminase_measurement,pQTLsmr_Adaptive NK cells
TNFRSF18,1,TNFRSF18,1203508,1:1217733:G:A,1,1217733,A,G,0.268278,⋯,-0.721839,0.0438161,5.615836e-61,-0.0131608,0.0256414,6.077665e-01,4.936223e-01,20,adenosine_deaminase_measurement,pQTLsmr_Adaptive NK cells
TNFRSF4,1,TNFRSF4,1211326,1:1217251:C:A,1,1217251,A,C,0.266509,⋯,-0.653871,0.0426808,5.620573e-53,0.004741,0.0321179,8.826488e-01,7.801704e-01,20,adenosine_deaminase_measurement,pQTLsmr_Adaptive NK cells
SDF4,1,SDF4,1216931,1:1232416:C:T,1,1232416,T,C,0.225236,⋯,-0.331675,0.040547,2.838453e-16,-0.051255,0.0624243,4.116036e-01,7.170661e-02,19,adenosine_deaminase_measurement,pQTLsmr_Adaptive NK cells
B3GALT6,1,B3GALT6,1232237,1:1240390:G:A,1,1240390,A,G,0.223467,⋯,-0.35543,0.0397387,3.747308e-19,-0.0548631,0.0582817,3.465283e-01,3.902540e-02,20,adenosine_deaminase_measurement,pQTLsmr_Adaptive NK cells
CPTP,1,CPTP,1324756,1:1363031:C:T,1,1363031,C,T,0.17158,⋯,0.810526,0.0464073,2.624344e-68,-0.020974,0.0286486,4.640986e-01,1.843850e-01,12,adenosine_deaminase_measurement,pQTLsmr_Adaptive NK cells
MRPL20-AS1,1,MRPL20-AS1,1399515,1:1399922:C:T,1,1399922,T,C,0.113208,⋯,-0.561624,0.0538927,1.985297e-25,0.0254619,0.0392483,5.165069e-01,6.575267e-01,15,adenosine_deaminase_measurement,pQTLsmr_Adaptive NK cells


In [21]:
colnames(all_smr)

[1] "probeID"    "ProbeChr"   "Gene"       "Probe_bp"   "topSNP"    
 [6] "topSNP_chr" "topSNP_bp"  "A1"         "A2"         "Freq"      
[11] "b_GWAS"     "se_GWAS"    "p_GWAS"     "b_eQTL"     "se_eQTL"   
[16] "p_eQTL"     "b_SMR"      "se_SMR"     "p_SMR"      "p_HEIDI"   
[21] "nsnp_HEIDI" "disease"    "celltype"

In [22]:
all_smr$p_GWAS <- as.numeric(all_smr$p_GWAS)
all_smr$p_eQTL <- as.numeric(all_smr$p_eQTL)
all_smr$p_SMR <- as.numeric(all_smr$p_SMR)
all_smr <- all_smr %>% group_by(disease) %>% mutate(q_SMR = p.adjust(p_SMR, method = "BH")) %>% ungroup()
all_smr$p_HEIDI <- as.numeric(all_smr$p_HEIDI)

In [23]:
sub_smr_all <- subset(all_smr, p_GWAS < 0.00001 & p_HEIDI > 0.01 & q_SMR < 0.05 & p_eQTL < 0.00000005)
sub_smr_all

probeID,ProbeChr,Gene,Probe_bp,topSNP,topSNP_chr,topSNP_bp,A1,A2,Freq,⋯,se_eQTL,p_eQTL,b_SMR,se_SMR,p_SMR,p_HEIDI,nsnp_HEIDI,disease,celltype,q_SMR
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<dbl>
ADA,20,ADA,44584896,20:44672026:G:A,20,44672026,A,G,0.248821,⋯,0.047633,7.888813e-11,0.387417,0.0761698,3.652498e-07,0.07737781,15,adenosine_deaminase_measurement,pQTLsmr_CD56bright NK cells,1.435087e-03
ADA,20,ADA,44584896,20:44672026:G:A,20,44672026,A,G,0.248821,⋯,0.0445726,2.169461e-16,0.327829,0.0566241,7.056788e-09,0.10311220,17,adenosine_deaminase_measurement,pQTLsmr_CD8+ Temra,6.223835e-05
ADA,20,ADA,44584896,20:44672026:G:A,20,44672026,A,G,0.248821,⋯,0.0430986,4.858822e-15,0.355572,0.0629222,1.595318e-08,0.12394730,16,adenosine_deaminase_measurement,pQTLsmr_GZMB+ CD8+ Tem,8.904902e-05
ADA,20,ADA,44584896,20:44672026:G:A,20,44672026,A,G,0.248821,⋯,0.037901,7.806166e-09,0.548478,0.11637,2.438305e-06,0.30122060,11,adenosine_deaminase_measurement,pQTLsmr_IRF1- GBP2- NDNs,7.284236e-03
CTSL,9,CTSL,87724051,9:87709162:C:T,9,87709162,T,C,0.366745,⋯,0.0444547,5.643760e-26,0.131681,0.0279213,2.403523e-06,0.03521279,13,beta_nerve_growth_factor_measurement,pQTLsmr_CD27+ Th17,4.248768e-02
CTSL,9,CTSL,87724051,9:87725948:C:A,9,87725948,A,C,0.302476,⋯,0.041719,2.241712e-30,0.151301,0.0281894,7.992178e-08,0.04550725,8,beta_nerve_growth_factor_measurement,pQTLsmr_Non-classical monocytes,5.651189e-03
CTSL,9,CTSL,87724051,9:87709162:C:T,9,87709162,T,C,0.366745,⋯,0.0414953,6.321745e-78,0.0795696,0.0156779,3.869217e-07,0.01282709,16,beta_nerve_growth_factor_measurement,pQTLsmr_Tfh,9.119615e-03
CTLA4,2,CTLA4,203853888,2:203852990:G:T,2,203852990,T,G,0.202241,⋯,0.042559,2.319162e-12,-0.208375,0.0510714,4.502263e-05,0.22332970,9,C_C_motif_chemokine_19_measurement,pQTLsmr_CD27+ Th17,1.906911e-02
LTB,6,LTB,31580525,6:31550577:G:A,6,31550577,G,A,0.251179,⋯,0.0298767,1.535739e-14,0.285294,0.0651807,1.203424e-05,0.07710242,20,C_C_motif_chemokine_19_measurement,pQTLsmr_Core classical monocytes,6.900397e-03


In [24]:
table(sub_smr_all$disease)


                                          adenosine_deaminase_measurement 
                                                                        4 
                                     beta_nerve_growth_factor_measurement 
                                                                        3 
                                       C_C_motif_chemokine_19_measurement 
                                                                        6 
                                   C_C_motif_chemokine_4_like_measurement 
                                                                       16 
                                     C_X_C_motif_chemokine_10_measurement 
                                                                        3 
                                      C_X_C_motif_chemokine_6_measurement 
                                                                        1 
                                                         CCL2_measurement 
                        

In [25]:
write.csv(sub_smr_all,'/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Result/SMR_eQTL_pQTL_2.csv')